In [36]:
import re
import pandas as pd
import numpy as np

years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]  #cn.years_annual

##### Functions to carry over to geospatial implementation

In [37]:
""" Regex-based land-use reclassification rules
These rules replace the default land use classes.
1. Convert annual GLAD LC values to LU tokens.
2. Use regex to identify token patterns for exceptions.
3. Reclassify token arrays and assign a matching node_code array.

Node codes used here:
1) Settlements and Infrastructure:
    10 = Built from GLAD data

2) Cropland:
    20  = Crop from GLAD data
    21  = Crop from oil palm extent
    22  = Crop from SDPT tree crop extent
    23X = Crop from permanent agriculture driver
         - 233 = TV after TCL where driver is permanent agriculture (assume tree crops)
         - 234 = SV after TCL outside GPW extent (i.e. "rangeland") where driver is permanent agriculture (assume crops)

3) Forest:
    30  = Tall veg from GLAD data
    31  = Forest from SDPT planted forest extent
    32  = Forest from GMW mangrove extent
    33X = Short vegetation or bare reclassified as Forest using drivers rules (assume unstocked forest)
        333 = Forest from shifting cultivation driver
        334 = Forest from logging driver
        335 = Forest from wildfire driver
        337 = Forest from natural disturbance driver

4) Grassland:
    40 = Short veg from GLAD data
    41 = Short veg from permanent agriculture driver
        - SV after TCL inside GPW extent where driver is permanent agriculture (assume rangeland)


5) Wetland:
    50 = Wetland from GLAD data

6) Other
    60 = Bare from GLAD data
    61 = Water from GLAD data
    62 = Snow/ice from GLAD data
"""

# Settlements > Cropland > Forest Land > Grassland > Wetlands > Other
# Default GLAD LC numeric values
settlement_lc   = {250}                                         # Built up
cropland_lc     = {244}                                         # Cropland
forest_lc       = set(range(27, 49)) | set(range(127, 149))     # Tall vegetation
grass_lc        = set(range(5, 27)) | set(range(105, 127))      # Short veg
wetland_lc      = set(range(200, 205))                          # Wetland
bare_lc         = set(range(0, 5)) | set(range(100, 105))       # Bare
water_lc        = set(range(205, 208)) | {254}                  # Open water
ice_lc          = {241}                                         # Snow/ice

# Lookup table to go from GLAD LC code -> default LU token
lc_token_map = {
    **{v: "S" for v in settlement_lc},
    **{v: "C" for v in cropland_lc},
    **{v: "F" for v in forest_lc},
    **{v: "G" for v in grass_lc},
    **{v: "W" for v in wetland_lc},
    **{v: "B" for v in bare_lc},
    **{v: "O" for v in water_lc},
    **{v: "I" for v in ice_lc},
}

# Function to get land use token per land cover numeric value (tokens used for regex exception rules)
def token_for_lc(v):
    if v not in lc_token_map:
        raise ValueError(f"Unknown GLCLU code: {v}")
    return lc_token_map[v]

# Node code values based on what exception was applied
node_code_map = {
    "built_glad": 10,

    "crop_glad": 20,
    "crop_oil_palm": 21,
    "crop_sdpt_tree_crop": 22,
    "crop_perm_ag_driver": 23,

    "forest_glad": 30,
    "forest_gmw_mangrove": 31,
    "forest_sdpt_planted_forest": 32,
    "forest_shift_cult_driver": 333,
    "forest_logging_driver": 334,
    "forest_wildfire_driver": 335,
    "forest_nat_dist_driver": 337,

    "grass_glad": 40,
    "grass_gpw": 41,

    "wetland_glad": 50,

    "bare_glad": 60,
    "water_glad": 61,
    "ice_glad": 62,
}

# Default node codes before rules are applied
def default_node_code(token):
    if token == "S":
        return node_code_map["built_glad"]
    if token == "C":
        return node_code_map["crop_glad"]
    if token == "F":
        return node_code_map["forest_glad"]
    if token == "G":
        return node_code_map["grass_glad"]
    if token == "W":
        return node_code_map["wetland_glad"]
    if token == "B":
        return node_code_map["bare_glad"]
    if token == "O":
        return node_code_map["water_glad"]
    if token == "I":
        return node_code_map["ice_glad"]
    return None

# Function to override default values based on regex rules
def set_tokens(tokens, node_codes, indices, new_token, node_code):
    for i in indices:
        tokens[i] = new_token
        node_codes[i] = node_code

# Converts char tokens to final int values in LU map
lu_token_map = {
    "S": 1,
    "C": 2,
    "F": 3,
    "G": 4,
    "W": 5,
    "B": 6,
    "O": 6,
    "I": 6,
}

In [38]:
# TODO: Add oil palm planting year logic
def apply_extent_rules(lu_dict):
    tokens = lu_dict["tokens"]
    node_codes = lu_dict["node_codes"]

    crop_reclass_idx = [i for i, token in enumerate(tokens) if token in {"F", "G", "W", "B"}]
    forest_reclass_idx = [i for i, token in enumerate(tokens) if token in {"G", "W", "B"}]
    # TODO: May want to consider not including wetland?

    # Crop is highest priority and extents are applied in this order: oil palm -> SDPT tree crop --> pre-2000 plantation
    if lu_dict["oil_palm"]:
        set_tokens(tokens, node_codes, crop_reclass_idx, "C", node_code_map["crop_oil_palm"])
        return True
    if lu_dict["sdpt_tree_crop"]:
        set_tokens(tokens, node_codes, crop_reclass_idx, "C", node_code_map["crop_sdpt_tree_crop"])
        return True

    # If no crop extent applies, forest extents are applied by this order: GMW mangrove -> SDPT planted forest
    if lu_dict["gmw_mangrove"]:
        set_tokens(tokens, node_codes, forest_reclass_idx, "F", node_code_map["forest_gmw_mangrove"])
        return True
    if lu_dict["sdpt_planted_forest"]:
        set_tokens(tokens, node_codes, forest_reclass_idx, "F", node_code_map["forest_sdpt_planted_forest"])
        return True
    return False

In [39]:
# Tall vegetation all years
def apply_all_tall_veg(lu_dict):
    tcl_prior = lu_dict["tcl_prior"]
    driver = lu_dict["driver"]

    all_idx = range(len(lu_dict["tokens"]))

    # If TCL has occurred by the start of timeseries and the driver is permanent ag, assume tall veg is tree crops
    if tcl_prior and driver == 1:
        set_tokens(lu_dict["tokens"], lu_dict["node_codes"], all_idx, "C", node_code_map["crop_perm_ag_driver"])


In [40]:
# Short vegetation all years
def apply_all_short_veg(lu_dict):
    tcl_prior = lu_dict["tcl_prior"]
    driver = lu_dict["driver"]

    all_idx = range(len(lu_dict["tokens"]))

    driver_to_forest_node = {
        3: node_code_map["forest_shift_cult_driver"],
        4: node_code_map["forest_logging_driver"],
        5: node_code_map["forest_wildfire_driver"],
        7: node_code_map["forest_nat_dist_driver"],
    }

    # If TCL has occurred by the start of the timeseries and the driver is permanent ag and not in cultivated grass extent, assume crop
    if tcl_prior and driver == 1:
        if not lu_dict["gpw_cultiv_grass"]:
            set_tokens(lu_dict["tokens"], lu_dict["node_codes"], all_idx, "C", node_code_map["crop_perm_ag_driver"])
        else:
            set_tokens(lu_dict["tokens"], lu_dict["node_codes"], all_idx, "G", node_code_map["grass_gpw"])
    # If TCL has occurred by the start of the timeseries and the driver is shifting cultivation, logging, wildfire, or other natural disturbances, assume unstocked forest
    elif tcl_prior and driver in driver_to_forest_node:
        set_tokens(lu_dict["tokens"], lu_dict["node_codes"], all_idx, "F", driver_to_forest_node[driver])


In [41]:
def apply_regex_rules(lc_timeseries, driver, tcl_year, oil_palm, sdpt_tree_crop, sdpt_planted_forest, gmw_mangrove, gpw_cultiv_grass):

    # Create default token array and default node code array from LC timeseries
    tokens = [token_for_lc(v) for v in lc_timeseries]               #char array representing land use timeseries
    node_codes = [default_node_code(token) for token in tokens]     #int array representing class definition rules applied throughout the timeseries

    lu_dict ={
        "tokens": tokens,
        "node_codes": node_codes,
        "driver": driver,
        "tcl_year": tcl_year,
        "tcl_prior": (tcl_year != 0 and tcl_year <= min(years)),
        "oil_palm": oil_palm,
        "sdpt_tree_crop": sdpt_tree_crop,
        "sdpt_planted_forest": sdpt_planted_forest,
        "gmw_mangrove": gmw_mangrove,
        "gpw_cultiv_grass": gpw_cultiv_grass,
    }

    # Check if oil palm, tree crop or forest based on special cases
    extent_rule_applied = apply_extent_rules(lu_dict)

    if not extent_rule_applied:
        token_seq = "".join(lu_dict["tokens"]) #Creates a concat string
        if re.fullmatch(r"F+", token_seq):
            apply_all_tall_veg(lu_dict)
        elif re.fullmatch(r"G+", token_seq):
            apply_all_short_veg(lu_dict)

    # Final token and node code timeseries
    final_tokens = lu_dict["tokens"]
    node_code_ts = lu_dict["node_codes"]

    # Convert final tokens to numeric LU codes
    lu_ts = [lu_token_map[token] for token in final_tokens]

    # Create transition timeseries: 2015_2016 through 2023_2024
    transition_ts = [int(f"{lu_ts[i]}{lu_ts[i + 1]}") for i in range(len(lu_ts) - 1)]

    # Create sequential unique LU summary ([3, 3, 3, 4, 4, 4, 2, 2, 2] -> [3, 4, 2] -> Forest to Grass to Crop)
    summary = []
    for lu in lu_ts:
        if not summary or lu != summary[-1]:
            summary.append(lu)

    return lu_ts, node_code_ts, transition_ts, summary


##### Functions to run IPCC land use classification on tabular data

In [42]:
# Dictionaries to convert numeric output to text for export csv
driver_code_map = {
    1: "permanent_agriculture",
    2: "hard_commodities",
    3: "shifting_cultivation",
    4: "logging",
    5: "wildfire",
    6: "settlements_infrastructure",
    7: "other_natural_disturbances"
}

lu_code_map = {
    1: "settlements_infrastructure",
    2: "cropland",
    3: "forest",
    4: "grassland",
    5: "wetland",
    6: "other",
}

# Reverse lookup because node_code_map is name -> code, but export needs code -> name.
node_code_text_map = {v: k for k, v in node_code_map.items()}

# Converts input csv values as boolean values for classification rules
def as_bool(v):
    if pd.isna(v):
        return False
    if isinstance(v, str):
        v = v.strip().lower()
        if v in {"", "false", "f", "no", "n", "0"}:
            return False
        if v in {"true", "t", "yes", "y", "1"}:
            return True
        return False
    return bool(v)

# Converts input csv values as int values for classification rules
def as_int_or_0(v):
    if pd.isna(v):
        return int(0)
    try:
        return int(v)
    except (TypeError, ValueError):
        return int(0)

# Iterate through each scenario and classify LC to create LU timeseries
def classify_dataframe(df):
    out = []
    lc_cols = [f"lc_{y}" for y in years]
    for idx, row in df.iterrows():
        scenario_id = row.get("id", idx)
        lc_ts=[as_int_or_0(row.get(lc)) for lc in lc_cols]        #Array of ints
        driver=as_int_or_0(row.get("driver"))                               #Int
        tcl_year=as_int_or_0(row.get("tcl_year"))                           #Int
        oil_palm=as_bool(row.get("oil_palm"))                           #Boolean
        sdpt_tree_crop=as_bool(row.get("sdpt_tree_crop"))               #Boolean
        sdpt_planted_forest=as_bool(row.get("sdpt_planted_forest"))     #Boolean
        gmw_mangrove=as_bool(row.get("gmw_mangrove"))                   #Boolean
        gpw_cultiv_grass=as_bool(row.get("gpw_cultiv_grass"))           #Boolean

        lu_ts, node_code_ts, transition_ts, summary = apply_regex_rules(lc_ts, driver, tcl_year, oil_palm, sdpt_tree_crop,
                                                                        sdpt_planted_forest, gmw_mangrove, gpw_cultiv_grass)

        # Convert numeric values to text values in export columns
        # Land use timeseries
        lu_cols = {f"LU_{y}": lu_code_map.get(lu_ts[i], "unknown") for i, y in enumerate(years)}

        # Node code timeseries
        node_code_cols = {f"node_{y}": node_code_text_map.get(node_code_ts[i], "unknown") for i, y in enumerate(years)}

        # Land use transition timeseries (create column to flag whether a transition happened at all)
        conversion = False  # Land use transition flag
        trans_cols = {}

        for i, (a, b) in enumerate(zip(years[:-1], years[1:])):
            transition_code = int(transition_ts[i])

            from_code = transition_code // 10
            to_code = transition_code % 10

            from_lu = lu_code_map[from_code]
            to_lu = lu_code_map[to_code]

            key = f"LU_{a}_{b}"

            if from_code == to_code:
                trans_cols[key] = f"{from_lu} remaining {to_lu}"
            else:
                trans_cols[key] = f"{from_lu} to {to_lu}"
                conversion = True

        # Summary all LU transitions that occurred during entire timeseries
        summary_text = " to ".join(lu_code_map.get(lu, str(lu))for lu in summary)

        out.append({
            "id": scenario_id,
            "tcl_year": (np.nan if tcl_year == 0 else tcl_year),
            "driver": (np.nan if driver == 0 else driver_code_map.get(driver, str(driver))),
            "oil_palm": True if oil_palm else np.nan,
            "sdpt_tree_crop": True if sdpt_tree_crop else np.nan,
            "sdpt_planted_forest": True if sdpt_planted_forest else np.nan,
            "gmw_mangrove": True if gmw_mangrove else np.nan,
            "gpw_cultiv_grass": True if gpw_cultiv_grass else np.nan,
            **lu_cols,
            **node_code_cols,
            **trans_cols,
            "summary": summary_text,
            "conversion_occurred": conversion,
        })

    return pd.DataFrame(out)

##### Read in scenarios from spreadsheet and export the resulting land use classification spreadsheet

In [44]:
# Read in data
file_path = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/LUC/conversion_LUC_scenarios.xlsx"
sheet_name = "scenarios"
scenarios_df = pd.read_excel(file_path, sheet_name=sheet_name)

# Coerce to numeric
lc_cols = [f"lc_{y}" for y in years]
numeric_cols = lc_cols + ["driver", "tcl_year"]
for c in numeric_cols:
    if c in scenarios_df.columns:
        scenarios_df[c] = pd.to_numeric(scenarios_df[c], errors="coerce")

# Run land use classification
results_df = classify_dataframe(scenarios_df)

# Export results to xlsx
results_df.to_excel("/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/LUC/conversion_LUC_scenario_results.xlsx", index=False)

# Print results
print("\nClassification Results:")
print(results_df)


Classification Results:
      id  tcl_year                      driver oil_palm sdpt_tree_crop  \
0    1.0       NaN                         NaN     True            NaN   
1    2.0       NaN                         NaN      NaN           True   
2    3.0       NaN                         NaN      NaN            NaN   
3    4.0       NaN                         NaN      NaN            NaN   
4    5.0    2015.0       permanent_agriculture      NaN            NaN   
5    6.0    2015.0       permanent_agriculture      NaN            NaN   
6    7.0    2020.0       permanent_agriculture      NaN            NaN   
7    8.0    2020.0       permanent_agriculture      NaN            NaN   
8    9.0    2015.0        shifting_cultivation      NaN            NaN   
9   10.0    2015.0                     logging      NaN            NaN   
10  11.0    2015.0                    wildfire      NaN            NaN   
11  12.0    2015.0  other_natural_disturbances      NaN            NaN   
12  13.0    2